# 🟡 Colab E-i — TensorFlow Low-Level from Scratch
**tf.Variable + tf.GradientTape + tf.function — NO Keras layers**

In [ ]:
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
tf.random.set_seed(42); np.random.seed(42)
print('TF:', tf.__version__)

## 📊 Section 1 — Data

In [ ]:
N=1000
x1=np.random.uniform(-2,2,(N,1)); x2=np.random.uniform(-2,2,(N,1)); x3=np.random.uniform(-2,2,(N,1))
y=2*x1**2+3*x2*x3+np.sin(x1*x2)+0.5*x3**2+np.random.normal(0,0.1,(N,1))
X=np.hstack([x1,x2,x3])
X_n=(X-X.mean(0))/X.std(0); y_mean,y_std=float(y.mean()),float(y.std()); y_n=(y-y_mean)/y_std
sp=int(0.8*N)
def tf32(a): return tf.constant(a,dtype=tf.float32)
Xtr,Xte=tf32(X_n[:sp]),tf32(X_n[sp:]); Ytr,Yte=tf32(y_n[:sp]),tf32(y_n[sp:])
print(Xtr.shape, Ytr.shape)

## ⚙️ Section 2 — tf.Variable Parameters

In [ ]:
def he_var(fan_in, fan_out, name):
    return tf.Variable(tf.random.normal([fan_in,fan_out])*(2/fan_in)**0.5, name=name, trainable=True)

W1=he_var(3,64,'W1');  b1=tf.Variable(tf.zeros([1,64]),name='b1')
W2=he_var(64,32,'W2'); b2=tf.Variable(tf.zeros([1,32]),name='b2')
W3=he_var(32,16,'W3'); b3=tf.Variable(tf.zeros([1,16]),name='b3')
W4=he_var(16,1,'W4');  b4=tf.Variable(tf.zeros([1,1]),name='b4')
tvars=[W1,b1,W2,b2,W3,b3,W4,b4]
print(f'Parameters: {sum(tf.size(v).numpy() for v in tvars)}')

## ➡️ Section 3 — Forward Pass & GradientTape Training

In [ ]:
@tf.function
def forward(X):
    A1 = tf.nn.relu(tf.matmul(X,W1) + b1)
    A2 = tf.nn.relu(tf.matmul(A1,W2) + b2)
    A3 = tf.nn.tanh(tf.matmul(A2,W3) + b3)
    return tf.matmul(A3,W4) + b4

# Adam state
ms=[tf.Variable(tf.zeros_like(v)) for v in tvars]
vs=[tf.Variable(tf.zeros_like(v)) for v in tvars]
step=tf.Variable(0,dtype=tf.float32)
lr,b1c,b2c,eps=1e-3,0.9,0.999,1e-8

@tf.function
def train_step(Xb, Yb):
    step.assign_add(1)
    with tf.GradientTape() as tape:
        pred = forward(Xb)
        loss = tf.reduce_mean((pred-Yb)**2)
    grads = tape.gradient(loss, tvars)
    for i,(g,v) in enumerate(zip(grads,tvars)):
        ms[i].assign(b1c*ms[i]+(1-b1c)*g)
        vs[i].assign(b2c*vs[i]+(1-b2c)*g*g)
        mh=ms[i]/(1-b1c**step)
        vh=vs[i]/(1-b2c**step)
        v.assign_sub(lr*mh/(tf.sqrt(vh)+eps))
    return loss

print('Train step compiled')

## 🏋️ Section 4 — Training Loop

In [ ]:
EPOCHS,BATCH=1000,64; history=[]
ds=(tf.data.Dataset.from_tensor_slices((Xtr,Ytr))
       .shuffle(800).batch(BATCH))

for epoch in range(EPOCHS):
    for Xb,Yb in ds:
        loss=train_step(Xb,Yb)
    history.append(float(loss))
    if (epoch+1)%100==0:
        vl=tf.reduce_mean((forward(Xte)-Yte)**2).numpy()
        print(f'Ep {epoch+1:4d} | Loss: {float(loss):.5f} | Val: {vl:.5f}')

## 📈 Section 5 — Results

In [ ]:
plt.figure(figsize=(10,4))
plt.semilogy(history,'steelblue',lw=1.5); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Colab E-i — TF Scratch'); plt.grid(alpha=0.3); plt.show()

yp=forward(Xte).numpy()*y_std+y_mean; yt=Yte.numpy()*y_std+y_mean
plt.figure(figsize=(6,6))
plt.scatter(yt,yp,alpha=0.4,s=15,color='steelblue')
mn,mx=yt.min(),yt.max(); plt.plot([mn,mx],[mn,mx],'r--',lw=2)
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Colab E-i — Predicted vs Actual'); plt.grid(alpha=0.3); plt.show()
print(f'Test MSE: {((yp-yt)**2).mean():.4f}')